# Run calibration

Outcomes:
- Prepare ACS demographic targets and ATUS conditional cluster distribution table
- Assign mobility users to ATUS behavioral clusters via k-NN on the precomputed distance matrix
- Run two-stage calibration (IPF demographic + behavioral raking) with replicate weights
- Export main and replicate weights

**Pipeline position:** follows `walkthrough_02_process_acs.ipynb` and `walkthrough_03_process_atus.ipynb`

In [1]:
import numpy as np
import pandas as pd

%load_ext autoreload
%autoreload 2
from helpers import prep_calibration_inputs

from mobcalibrate import Calibrator

In [ ]:
# ======================
# PATHS
# ======================
DATA_DIR = "data/processed"
CBSA_CODE = 38060

# Inputs from previous steps
DIST_MATRIX_FILE = f"{DATA_DIR}/distance_matrix_3cat_full_embedding.parquet"
ATUS_META_FILE   = f"{DATA_DIR}/atus_meta_{CBSA_CODE}.csv"
ACS_CBG_FILE     = f"{DATA_DIR}/acs_cbg_distr_{CBSA_CODE}.csv"
INCOME_MARGIN_FILE = f"{DATA_DIR}/acs_income_margins_{CBSA_CODE}.csv"
AGE_MARGIN_FILE    = f"{DATA_DIR}/acs_age_margins_{CBSA_CODE}.csv"
MEDOID_INFO_FILE = f"{DATA_DIR}/atus_medoid_info_{CBSA_CODE}.json"

# Outputs
OUT_CALIBRATION_RESULTS = f"{DATA_DIR}/calibration_results_{CBSA_CODE}.pkl"
OUT_WEIGHTS = f"{DATA_DIR}/weights_{CBSA_CODE}.parquet"

# ======================
# STRATIFICATION VARIABLES
# (must match the coding used in walkthrough_03)
# ======================
ROW_VAR          = "income"       # row stratification variable (matches atus_meta column)
COL_VAR          = "age"          # col stratification variable (matches atus_meta column)
CLUSTER_LABEL_COL = "cluster_label"
WEIGHT_COL        = "TUFINLWGT"
GEOID_COL         = "GEOID"

NUM_CLUSTERS = 4

# ======================
# KNN ASSIGNMENT
# ======================
KNN_K          = 10    # number of nearest ATUS neighbors
KNN_THRESHOLD  = 0.5   # min fraction of neighbors required to agree on a label

# ======================
# CALIBRATOR SETTINGS
# ======================
NUM_REPLICATES = 51
SEED           = 1234

## 1. Load data

In [3]:
atus_meta  = pd.read_csv(ATUS_META_FILE)
dist       = pd.read_parquet(DIST_MATRIX_FILE)
acs_cbg    = pd.read_csv(ACS_CBG_FILE, dtype={GEOID_COL: str})
row_margin = pd.read_csv(INCOME_MARGIN_FILE)
col_margin = pd.read_csv(AGE_MARGIN_FILE)
medoid_info = prep_calibration_inputs.load_medoid_info(MEDOID_INFO_FILE)

print(f"ATUS respondents: {len(atus_meta)}")
print(f"Mobility users:   {len(dist)}")
print(f"CBGs in ACS:      {len(acs_cbg)}")

ATUS respondents: 2033
Mobility users:   141530
CBGs in ACS:      2987


In [4]:
# hypothetical user ids (since real ones cannot be published due to data agreement)
dist['user_id'] = range(1, len(dist)+1)

# fix bug in distance matrix
dist['20190504191857'] = dist.loc[:, '20190504191857'].str.replace('\x18', '').astype(float)

## 2. Prepare ACS targets

`prep_acs_targets` normalizes the marginal counts into probability distributions
and extracts category label arrays. These are passed directly to the `Calibrator`.

In [5]:
acs_targets = prep_calibration_inputs.prep_acs_targets(
    row_margin_df = row_margin,
    col_margin_df = col_margin,
    row_var = "hh_income",
    col_var = "age_group",
)

print("Row categories (income):", acs_targets["row_cats"])
print("Col categories (age):   ", acs_targets["col_cats"])
print("Target population:      ", acs_targets["target_pop_tot"])

Row categories (income): ['<35k' '35k-75k' '75k-125k' '125k+']
Col categories (age):    ['18-24' '25-44' '45-66' '67+']
Target population:       3708148


## 3. Prepare ATUS behavioral target table

Computes P(cluster | demographic stratum) from ATUS respondent weights.
Rows index joint demographic strata (income × age), columns index clusters.
Each row sums to 1.

In [6]:
atus_target_P = prep_calibration_inputs.prep_atus_target(
    atus_meta_df      = atus_meta,
    row_var           = ROW_VAR,
    col_var           = COL_VAR,
    cluster_label_col = CLUSTER_LABEL_COL,
    weight_col        = WEIGHT_COL,
    num_row_cats      = acs_targets["num_row_cats"],
    num_col_cats      = acs_targets["num_col_cats"],
    num_clusters      = NUM_CLUSTERS,
)

print(f"Target table shape: {atus_target_P.shape}  (strata * clusters)")
atus_target_P

Target table shape: (16, 4)  (strata * clusters)


array([[0.42763161, 0.03429583, 0.23827778, 0.29979478],
       [0.36993845, 0.11836833, 0.31559443, 0.19609879],
       [0.36191645, 0.22859029, 0.21176841, 0.19772486],
       [0.40904004, 0.32358656, 0.04104545, 0.22632795],
       [0.378268  , 0.06594847, 0.25098553, 0.304798  ],
       [0.20770882, 0.07507449, 0.44093081, 0.27628587],
       [0.3473284 , 0.1097163 , 0.30674674, 0.23620855],
       [0.39891894, 0.22335438, 0.02516295, 0.35256373],
       [0.36988238, 0.20372292, 0.26291417, 0.16348054],
       [0.22803558, 0.06123894, 0.4485094 , 0.26221608],
       [0.30067875, 0.0745933 , 0.3461343 , 0.27859365],
       [0.63952756, 0.13966989, 0.00989712, 0.21090544],
       [0.37892297, 0.        , 0.23703972, 0.38403731],
       [0.1876211 , 0.06620544, 0.50547551, 0.24069795],
       [0.25635734, 0.10610764, 0.39806598, 0.23946903],
       [0.44594546, 0.20819012, 0.07414072, 0.2717237 ]])

## 4. Assign mobility users to ATUS clusters

Uses k-NN voting on the precomputed distance matrix. Each mobility user is
assigned the plurality cluster among their `KNN_K` nearest ATUS neighbors,
provided that cluster accounts for at least `KNN_THRESHOLD` of those neighbors.
Users below the threshold, or farther than the medoid distance threshold for
their cluster, are marked unassigned (`-1`) and excluded from the behavioral
raking step (their demographic weights are still computed in stage 1).

In [7]:
assigned_labels = prep_calibration_inputs.assign_mobility_clusters(
    dist_df           = dist,
    atus_meta_df      = atus_meta,
    cluster_label_col = CLUSTER_LABEL_COL,
    k                 = KNN_K,
    threshold         = KNN_THRESHOLD,
    medoid_indices    = medoid_info['medoid_indices'],
    medoid_thresholds = medoid_info['medoid_thresholds'],
)

n_assigned   = int((assigned_labels >= 0).sum())
n_unassigned = int((assigned_labels == -1).sum())
print(f"Assigned:   {n_assigned}  ({n_assigned / len(assigned_labels):.1%})")
print(f"Unassigned: {n_unassigned}  ({n_unassigned / len(assigned_labels):.1%})")
#pd.Series(assigned_labels).value_counts().sort_index().rename("count")

100%|██████████| 141530/141530 [00:01<00:00, 97111.91it/s]

Assigned:   133830  (94.6%)
Unassigned: 7700  (5.4%)


## 5. Filter to users with valid home CBGs

Drops mobility users whose home GEOID does not appear in the ACS CBG table.
These users cannot be calibrated because no demographic distribution is available
for their home census block group.

In [8]:
user_ids, home_cbgs, assigned_labels, n_dropped = prep_calibration_inputs.filter_valid_users(
    users_df        = dist,
    acs_cbg_df      = acs_cbg,
    assigned_labels = assigned_labels,
    geoid_col       = GEOID_COL,
)

print(f"Users retained: {len(home_cbgs)}")
print(f"Users dropped (missing CBG): {n_dropped}")
print("Proportion of assigned cluster labels:")
pd.Series(assigned_labels).value_counts(normalize=True).sort_index().rename("proportion")

35151 rows have user GEOID missing from ACS GEOID (or null), e.g. ['040136103001' '040130506062' '040138171001' '040130405172'
 '040138118002']
Unique missing GEOIDs (excluding null): 504
Users retained: 106379
Users dropped (missing CBG): 35151
Proportion of assigned cluster labels:


-1    0.054766
 0    0.213322
 1    0.187979
 2    0.319556
 3    0.224377
Name: proportion, dtype: float64

In [9]:
print(user_ids[:5])
print(home_cbgs[:5])
print(assigned_labels[:5])

0    1
1    4
2    5
3    6
4    8
Name: user_id, dtype: int64
['040131112011' '040130925004' '040130715064' '040132168511'
 '040131167031']
[0 1 3 0 2]


## 6. Run calibration

Initializes the `Calibrator` and runs the two-stage procedure for the main weight
set and all replicates. Each replicate independently samples demographic codes from
the CBG-level ACS distributions before raking, propagating individual-level
demographic uncertainty into the weight variance.

`create_main_weights()` and `create_replicate_weights()` both use the same `mode`
argument, which controls which calibration stages are run. The default `"behavioural_full"`
runs both stage 1 (demographic IPF) and stage 2 (behavioral raking).

In [11]:
calibrator = Calibrator(
    unit_ids               = user_ids,
    home_cbgs              = home_cbgs,
    assigned_cluster_labels = assigned_labels,
    acs_cbg_probs_df       = acs_cbg,
    acs_row_var_name       = ROW_VAR,
    acs_col_var_name       = COL_VAR,
    acs_row_cats           = acs_targets["row_cats"],
    acs_col_cats           = acs_targets["col_cats"],
    acs_row_margin         = acs_targets["row_margin"],
    acs_col_margin         = acs_targets["col_margin"],
    atus_target_table      = atus_target_P,
    target_pop_tot         = acs_targets["target_pop_tot"],
    geoid_col              = GEOID_COL,
    num_replicates         = NUM_REPLICATES,
    seed                   = SEED,
)

In [17]:
CALIBRATION_MODE = "demographic_behavioral"
calibration_results = calibrator.create_weights(mode = CALIBRATION_MODE)

Weights created, returning CalibrationResult object.
Use all_final_weights() to obtain final weights.
Use to_df() to obtain replicate-specific stage1 and stage2 weights with metadata.
Use to_long_df() to obtain all replicate weights with metadata.



## 7. Inspect weights

In [14]:
calibration_results.all_final_weights()

,unit_id,weight_final_0,weight_final_1,weight_final_2,weight_final_3,weight_final_4,weight_final_5,weight_final_6,weight_final_7,weight_final_8,...,weight_final_41,weight_final_42,weight_final_43,weight_final_44,weight_final_45,weight_final_46,weight_final_47,weight_final_48,weight_final_49,weight_final_50
0,1,38,63,66,73,59,96,56,73,37,...,72,58,64,67,58,37,74,36,35,65
1,4,20,23,12,43,13,58,39,43,20,...,38,23,11,13,39,11,13,12,44,13
2,5,29,45,35,36,43,59,38,39,32,...,44,37,31,43,36,43,35,37,31,41
3,6,63,63,44,36,55,96,44,35,34,...,61,36,35,63,27,65,35,35,56,97
4,8,33,45,51,7,50,45,3,34,8,...,33,48,44,45,35,33,1,8,23,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106374,141526,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
106375,141527,63,29,66,27,43,37,56,58,63,...,36,57,37,36,36,45,28,57,43,28
106376,141528,12,38,12,15,20,24,18,24,20,...,20,12,43,18,38,18,12,11,58,24
106377,141529,25,15,15,12,44,20,12,15,45,...,20,15,15,25,59,26,24,11,12,26


In [ ]:
calibration_results.to_df(replicate_id=0) # replicate_id=0 => main weights

,unit_id,weight1,weight_final,sampled_income_code,sampled_age_code,sampled_joint_stratum_code,calibration_mode
0,1,38,38,1,1,5,demographic_behavioral
1,4,35,20,1,2,6,demographic_behavioral
2,5,33,29,2,3,11,demographic_behavioral
3,6,31,63,3,3,15,demographic_behavioral
4,8,31,33,2,2,10,demographic_behavioral
...,...,...,...,...,...,...,...
106374,141526,41,0,0,1,1,demographic_behavioral
106375,141527,31,63,3,3,15,demographic_behavioral
106376,141528,31,12,3,1,13,demographic_behavioral
106377,141529,41,25,0,1,1,demographic_behavioral


In [20]:
calibration_results.to_long_df()

,replicate_id,unit_id,weight1,weight_final,sampled_income_code,sampled_age_code,sampled_joint_stratum_code,calibration_mode
0,0,1,38,38,1,1,5,demographic_behavioral
1,0,4,35,20,1,2,6,demographic_behavioral
2,0,5,33,29,2,3,11,demographic_behavioral
3,0,6,31,63,3,3,15,demographic_behavioral
4,0,8,31,33,2,2,10,demographic_behavioral
...,...,...,...,...,...,...,...,...
5425324,50,141526,30,0,3,1,13,demographic_behavioral
5425325,50,141527,30,28,3,1,13,demographic_behavioral
5425326,50,141528,33,24,2,3,11,demographic_behavioral
5425327,50,141529,41,26,0,1,1,demographic_behavioral


## 8. Export

In [ ]:
# save entire object as pickle
calibration_results.save(OUT_CALIBRATION_RESULTS)
print(f"CalibrationResult object saved to: {OUT_CALIBRATION_RESULTS}")

# or save weights df directly as either csv or parquet
calibration_results.to_long_df().to_parquet(OUT_WEIGHTS)
print(f"Weights df saved to: {OUT_WEIGHTS}")


# load back results:
#calibration_results = CalibrationResult.load(OUT_CALIBRATION_RESULTS)